# 1. 输入格式化

对于复杂的对话（带图片或工具结果），建议使用 content_blocks 列表形式构建 HumanMessage或 AIMessage 。

借助 content_blocks ，我们可以用一套标准代码，无缝地在不同厂商的模型之间切换。

举例1： OpenAI模型

In [7]:
import base64
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

load_dotenv(override=True)

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL')
)

def encode_image(img_path: str):
    """
    将一张本地图片转换成 Base64 编码的 Data URI 字符串，方便再文本中嵌入图片数据

    Args:
        img_path: 图片路径，字符串参数

    Returns:
        返回为None

    """
    with open(img_path, 'rb') as img_file:
        return base64.b64encode(img_file.read()).decode('utf-8')

#图片的相对路径地址
img_url = "image_test.png"

base64_image = encode_image(img_url)

response = model_openai.invoke(
    [HumanMessage(
        content_blocks=[
            {'type': 'text', 'text': '这张图里有什么？'},
            {'type': 'image', 'base64': base64_image, 'mime_type': 'image/png'}
        ]
    )]
)

print(response.content)


这张图片展示了一瓶**雅诗兰黛（Estée Lauder）的粉底液**，从外观特征来看，这是该品牌非常著名的 **Double Wear 持妆粉底液（Double Wear Stay-in-Place Makeup）**。

以下是图片中的具体细节：

1.  **主体产品**：
    *   这是一个长方形的透明瓶身，里面装着**浅米色（或自然肤色/象牙色）**的液体粉底。
    *   瓶身正面印有金色的 **"ESTÉE LAUDER"** 品牌标志。
    *   标志下方有黑色的英文小字，虽然有些模糊，但通常是产品系列名称 "Double Wear Stay-in-Place Makeup" 以及容量信息（通常是35ml）。

2.  **包装设计**：
    *   瓶子顶部有一个**金色的泵头**（用于按压取用粉底）。
    *   泵头外面罩着一个**透明的方形盖子**，可以看到内部金色的机械结构。
    *   瓶身和盖子之间有一圈金色的装饰环，显得比较精致。

3.  **背景与光影**：
    *   背景是**暖米色/浅棕色**的纯色背景，与粉底液的颜色非常接近，营造出一种柔和、高级且统一的视觉风格（这是美妆产品摄影常用的手法）。
    *   光线从右侧照射过来，在瓶子的左侧投下了长长的阴影，增加了立体感。

总的来说，这是一张典型的雅诗兰黛 Double Wear 粉底液的产品宣传图或静物摄影。


# 2. 输出格式化
content_blocks 还可用于输出格式化，以deepseek官网的 deepseek-v4-flash 为例，其输出包含思考内容，后者位于 additional_kwargs 的 reasoning_content 字段下。比如：

举例2：

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url=os.getenv('DASHSCOPE_BASE_URL'),
    extra_body={"thinking": {"type": "disabled"}},
)

response = model_openai.invoke("你好，一句话回答")

print('=' * 20, '-> response <-', '=' * 20)
print(response)

print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)

print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)




==================== -> response <- ====================
content='你好！请问有什么我可以帮你的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 16, 'total_tokens': 25, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 16}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-flash', 'system_fingerprint': None, 'id': 'chatcmpl-672f15a3-d7d3-9831-a265-fc311b040210', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a09ff3-cd49-7cb3-a494-07bc97d301e4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 9, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
==================== -> response.content <- ====================
你好！请问有什么我可以帮你的吗？
==================== -> response.content_blocks <- ====================
[{'type': 'text', 'text': '你好！请问有什么我可以帮你的吗？'}]
